# Bronze Layer — Step 3: Verify Table Properties | Step 4: Apply RBAC

Run this notebook **after** the bronze SDP pipeline has completed at least one run.

**Prerequisites:** `healthcare.bronze.{claims, providers, diagnosis, cost}` must exist.

**Widgets:**
- `catalog` — Unity Catalog catalog name (default: `healthcare`)
- `schema` — Schema name (default: `bronze`)
- `etl_service_account` — Service account used by the SDP pipeline (gets INSERT only)
- `billing_analyst_group` — Group for billing analysts (gets SELECT only)

Set `etl_service_account` and `billing_analyst_group` to empty string to skip RBAC grants.

In [ ]:
dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")
dbutils.widgets.text("etl_service_account", "", "ETL service account (INSERT only)")
dbutils.widgets.text("billing_analyst_group", "", "Billing analyst group (SELECT only)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
ETL_SA = dbutils.widgets.get("etl_service_account").strip()
ANALYST_GROUP = dbutils.widgets.get("billing_analyst_group").strip()

TABLES = ["claims", "providers", "diagnosis", "cost"]
REQUIRED_PROPS = {
    "delta.enableChangeDataFeed": "true",
    "delta.logRetentionDuration": "interval 2190 days",
    "delta.deletedFileRetentionDuration": "interval 2190 days",
}

print(f"Target namespace: {CATALOG}.{SCHEMA}")
print(f"Tables to verify: {TABLES}")

## Step 3a — Verify tables exist

In [ ]:
missing = []
for table in TABLES:
    fqn = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        spark.sql(f"DESCRIBE TABLE {fqn}").collect()
        print(f"  ✓  {fqn} exists")
    except Exception as e:
        missing.append(fqn)
        print(f"  ✗  {fqn} MISSING — {e}")

if missing:
    raise RuntimeError(
        f"Tables not found: {missing}. "
        "Run the bronze SDP pipeline first, then re-run this notebook."
    )

## Step 3b — Verify HIPAA TBLPROPERTIES

Confirms `delta.enableChangeDataFeed`, `delta.logRetentionDuration`, and `delta.deletedFileRetentionDuration` are set on all four tables (required at creation per HIPAA 45 CFR § 164.316(b)(2)(i)).

In [ ]:
failures = []

for table in TABLES:
    fqn = f"{CATALOG}.{SCHEMA}.{table}"
    desc_rows = spark.sql(f"DESCRIBE EXTENDED {fqn}").collect()

    props = {
        row["col_name"]: row["data_type"]
        for row in desc_rows
        if row["col_name"] in REQUIRED_PROPS
    }

    for prop_key, expected_value in REQUIRED_PROPS.items():
        actual = props.get(prop_key)
        if actual != expected_value:
            failures.append(f"{fqn}: {prop_key} = '{actual}' (expected '{expected_value}')")
        else:
            print(f"  ✓  {fqn}: {prop_key} = {actual}")

if failures:
    raise AssertionError(
        "HIPAA TBLPROPERTIES not correctly set:\n" + "\n".join(f"  • {f}" for f in failures)
    )

print("\n✓ All HIPAA TBLPROPERTIES verified across all 4 bronze tables.")

## Step 3c — Verify audit columns are populated

In [ ]:
AUDIT_COLUMNS = ["_ingested_at", "_source_file", "_pipeline_run_id"]
audit_failures = []

for table in TABLES:
    fqn = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        sample = spark.sql(f"SELECT {', '.join(AUDIT_COLUMNS)} FROM {fqn} LIMIT 5").collect()
    except Exception as e:
        audit_failures.append(f"{fqn}: could not query audit columns — {e}")
        continue

    if not sample:
        audit_failures.append(f"{fqn}: table is empty — pipeline may not have processed any files")
        continue

    for col in AUDIT_COLUMNS:
        null_count = sum(1 for row in sample if row[col] is None)
        if null_count > 0:
            audit_failures.append(f"{fqn}.{col}: {null_count}/{len(sample)} sampled rows have NULL")
        else:
            print(f"  ✓  {fqn}.{col} = {sample[0][col]}")

if audit_failures:
    raise AssertionError(
        "Audit column failures:\n" + "\n".join(f"  • {f}" for f in audit_failures)
    )

print("\n✓ All audit columns (_ingested_at, _source_file, _pipeline_run_id) are populated.")

## Step 3d — Row count verification

Compares bronze table row counts against the source CSV files in the landing volume.

In [ ]:
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"
CSV_FILENAMES = {
    "claims": "claims_1000.csv",
    "providers": "providers_1000.csv",
    "diagnosis": "diagnosis.csv",
    "cost": "cost.csv",
}

count_results = []
count_failures = []

for table in TABLES:
    fqn = f"{CATALOG}.{SCHEMA}.{table}"
    table_count = spark.sql(f"SELECT COUNT(*) AS n FROM {fqn}").collect()[0]["n"]
    csv_path = f"{VOLUME_ROOT}/{table}/{CSV_FILENAMES[table]}"

    try:
        csv_count = spark.read.option("header", "true").csv(csv_path).count()
        status = "MATCH" if table_count == csv_count else "MISMATCH"
        if status == "MISMATCH":
            count_failures.append(f"{fqn}: table={table_count} rows, csv={csv_count} rows")
        count_results.append({"table": fqn, "table_rows": table_count, "csv_rows": csv_count, "status": status})
    except Exception as e:
        count_results.append({"table": fqn, "table_rows": table_count, "csv_rows": "N/A", "status": f"CSV not found — {e}"})

display(spark.createDataFrame(count_results))

if count_failures:
    raise AssertionError("Row count mismatches:\n" + "\n".join(f"  • {f}" for f in count_failures))

print("\n✓ Row counts match source CSVs.")

## Step 4 — Apply Unity Catalog RBAC

Implements least-privilege access from the product spec:
- **ETL service account** → `INSERT` only (pipeline writes; cannot read or delete)
- **Billing analyst group** → `SELECT` only (can query; cannot modify)
- **Public** → no access (default-deny)

Skip if widgets are empty.

In [ ]:
if not ETL_SA and not ANALYST_GROUP:
    print("No principals configured. Set 'etl_service_account' and/or 'billing_analyst_group' widgets to apply RBAC.")
else:
    rbac_statements = []

    for principal in filter(None, [ETL_SA, ANALYST_GROUP]):
        rbac_statements.append(f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `{principal}`")
        rbac_statements.append(f"GRANT USE SCHEMA ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO `{principal}`")

    for table in TABLES:
        fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table}`"
        if ETL_SA:
            rbac_statements.append(f"GRANT INSERT ON TABLE {fqn} TO `{ETL_SA}`")
        if ANALYST_GROUP:
            rbac_statements.append(f"GRANT SELECT ON TABLE {fqn} TO `{ANALYST_GROUP}`")

    grant_failures = []
    for stmt in rbac_statements:
        try:
            spark.sql(stmt)
            print(f"  ✓  {stmt}")
        except Exception as e:
            grant_failures.append({"statement": stmt, "error": str(e)})
            print(f"  ✗  {stmt} — {e}")

    if grant_failures:
        display(spark.createDataFrame(grant_failures))
        raise RuntimeError(
            f"{len(grant_failures)} GRANT statement(s) failed. "
            "Verify principal names exist in the workspace and that you have MANAGE privilege on the tables."
        )

    print(f"\n✓ RBAC applied: {len(rbac_statements)} GRANT statements executed.")

## Summary

In [ ]:
print("=" * 60)
print("BRONZE LAYER VERIFICATION SUMMARY")
print("=" * 60)
print(f"  Catalog / Schema  : {CATALOG}.{SCHEMA}")
print(f"  Tables verified   : {', '.join(TABLES)}")
print(f"  TBLPROPERTIES     : ✓ HIPAA-compliant (CDF + interval 2190 days retention)")
print(f"  Audit columns     : ✓ _ingested_at, _source_file, _pipeline_run_id")
print(f"  Row counts        : ✓ Match source CSVs")
print(f"  RBAC              : {'✓ Applied' if (ETL_SA or ANALYST_GROUP) else '⚠ Skipped (no principals configured)'}")
print("=" * 60)
print("Next step: run bronze_profiling.ipynb")